In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

print(api_key[:10] + "..." if api_key else "API Key not found")


AQ.Ab8RN6J...


In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

response = llm.invoke("Say hello in one sentence.")

print(response.content)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Hello there!


In [21]:
%pip install -U"Unstructured[all-docs]"
%pip install -Uq langchain_chroma
%pip install -Uq langchain langchain-community langchain-openai



Note: you may need to restart the kernel to use updated packages.



[optparse.groups]Usage:[/]   
  c:\Users\cheta\AppData\Local\Programs\Python\Python313\python.exe -m pip install \[options] <requirement specifier> \[package-index-options] ...
  c:\Users\cheta\AppData\Local\Programs\Python\Python313\python.exe -m pip install \[options] -r <requirements file> \[package-index-options] ...
  c:\Users\cheta\AppData\Local\Programs\Python\Python313\python.exe -m pip install \[options] [-e] <vcs project url> ...
  c:\Users\cheta\AppData\Local\Programs\Python\Python313\python.exe -m pip install \[options] [-e] <local project path> ...
  c:\Users\cheta\AppData\Local\Programs\Python\Python313\python.exe -m pip install \[options] <archive url/path> ...

no such option: -n


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
%pip install pymupdf
%pip install pillow
%pip install docling
%pip install pdfplumber
%pip install tqdm

In [2]:
from pathlib import Path
from docling.document_converter import DocumentConverter

import fitz
import pdfplumber
from tqdm import tqdm
import os
print(os.getcwd())


d:\GenAiFinanceAnlyzer\GenAi-Finance_Analyzer


In [3]:
pdf_path = Path("data/reports/NYSE_HRL_2025.pdf")
converter=DocumentConverter()
result=converter.convert(pdf_path,page_range=(1,20))

[INFO] 2026-06-28 14:59:13,616 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-06-28 14:59:13,714 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\cheta\AppData\Local\Programs\Python\Python313\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-06-28 14:59:13,720 [RapidOCR] main.py:63: Using C:\Users\cheta\AppData\Local\Programs\Python\Python313\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-06-28 14:59:14,466 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-06-28 14:59:14,486 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\cheta\AppData\Local\Programs\Python\Python313\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-06-28 14:59:14,490 [RapidOCR] main.py:63: Using C:\Users\cheta\AppData\Local\Programs\Python\Python313\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-06-28 14:59:14,913 [RapidOCR] base.py:23: 

In [11]:
type(result)

docling.datamodel.document.ConversionResult

In [31]:

type(result.document)
document=result.document
markdown=document.export_to_markdown()
print(markdown[:100])

NameError: name 'result' is not defined

In [19]:
output_dir=Path("data/processed/")
output_dir.mkdir(parents=True,exist_ok=True)
output_file=output_dir/"NYSE_HRL_2025.md"
with open(output_file,"w",encoding="utf-8") as f:
    f.write(markdown)

In [ ]:
doc_dict=document.export_to_dict()
type(doc_dict)
doc_dict.keys()

dict_keys(['schema_name', 'version', 'name', 'origin', 'furniture', 'body', 'groups', 'texts', 'pictures', 'tables', 'key_value_items', 'form_items', 'pages'])

In [ ]:
import json
print(json.dumps(doc_dict,indent=2)[:3000])

In [28]:
print(len(doc_dict.get("texts",[])))
print(len(doc_dict.get("tables",[])))
print(len(doc_dict.get("pictures",[])))

434
1
50


In [ ]:
##image extraction+OCR
%pip install paddleocr
%pip install paddlepaddle


In [ ]:
import fitz#open and read PDf
from pathlib import Path#handle file paths
from PIL import Image#save image files
import io#convert image bytes into images


In [5]:
pdf_path = Path("data/reports/NYSE_HRL_2025.pdf")
doc=fitz.open(pdf_path)
print(f"Total pages:{len(doc)}")

Total pages:93


In [6]:
output_folder=Path("data/extracted_images")
output_folder.mkdir(parents=True,exist_ok=True)

In [7]:
##exracting images 
image_count=0
for page_number in range(len(doc)):
    page=doc.load_page(page_number)
    image_list=page.get_images(full=True)
    print(f"Page {page_number+1}: {len(image_list)} image(s)")
    
    for image_index,img in enumerate(image_list):
        xref=img[0]
        base_image=doc.extract_image(xref)
        image_bytes=base_image["image"]
        image_ext=base_image["ext"]
        image=Image.open(io.BytesIO(image_bytes))
        image_name=f"page_{page_number+1}_img_{image_index+1}.{image_ext}"
        image.save(output_folder/image_name)
        image_count+=1
    print(f"\nTotal Images Extracted:{image_count}")

Page 1: 14 image(s)

Total Images Extracted:14
Page 2: 16 image(s)

Total Images Extracted:30
Page 3: 1 image(s)

Total Images Extracted:31
Page 4: 1 image(s)

Total Images Extracted:32
Page 5: 4 image(s)

Total Images Extracted:36
Page 6: 5 image(s)

Total Images Extracted:41
Page 7: 6 image(s)

Total Images Extracted:47
Page 8: 1 image(s)

Total Images Extracted:48
Page 9: 24 image(s)

Total Images Extracted:72
Page 10: 1 image(s)

Total Images Extracted:73
Page 11: 0 image(s)

Total Images Extracted:73
Page 12: 0 image(s)

Total Images Extracted:73
Page 13: 0 image(s)

Total Images Extracted:73
Page 14: 0 image(s)

Total Images Extracted:73
Page 15: 0 image(s)

Total Images Extracted:73
Page 16: 0 image(s)

Total Images Extracted:73
Page 17: 0 image(s)

Total Images Extracted:73
Page 18: 0 image(s)

Total Images Extracted:73
Page 19: 0 image(s)

Total Images Extracted:73
Page 20: 0 image(s)

Total Images Extracted:73
Page 21: 0 image(s)

Total Images Extracted:73
Page 22: 0 image(s)

In [15]:
%pip install google-genai
%pip install langchain-google-genai
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: google-genai in c:\Users\cheta\AppData\Local\Programs\Python\Python313\Lib\site-packages (2.9.0)




[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import os
import google.genai as genai
from dotenv import load_dotenv

In [ ]:
load_dotenv()
api_key=os.getenv("GOOGLE_API_KEY")
client = genai.Client()

# Generate content using the new client syntax
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Hello! Give me a quick one-sentence research tip.",
)

print(response.text)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Always critically evaluate the credibility and potential bias of your sources to ensure reliable information.


In [ ]:
image=Image.open("data/extracted_images/page_8_img_1.png")
prompt="""You are a financial analyst.

Analyze this image from a company's annual report.

Identify:

1. Image type (bar chart, pie chart, graph, diagram, table, logo)

2. Important financial insights

3. Trends

4. Numbers

5. Axes labels

6. Years

7. Anything useful for answering future financial questions.

Return a concise but detailed description."""

print(response.text)

Here's an analysis of the image from a financial analyst's perspective:

1.  **Image Type:** Operational Photograph / Illustrative Image. It is a visual representation of the company's production facilities and workforce, rather than a data visualization tool.

2.  **Important Financial Insights (Indirect):**
    *   **Operational Activity:** The image confirms active manufacturing/production operations, indicating the company has tangible assets (plant, machinery) and ongoing revenue-generating activities.
    *   **Labor Force:** A significant human capital component is evident, implying substantial labor costs (wages, benefits) as part of the Cost of Goods Sold (COGS) or operational expenses.
    *   **Capital Investment:** The presence of industrial machinery and a structured facility suggests past and ongoing Capital Expenditure (CapEx) in property, plant, and equipment.
    *   **Product Focus:** The clear depiction of food products (sausages) being processed defines the company'

In [28]:
output_folder=Path("data/extracted_text")
output_folder.mkdir(parents=True,exist_ok=True)
image_folder=Path("data/extracted_images")


In [29]:
for image_path in image_folder.iterdir():
    if image_path.suffix.lower() not in [".jpg",".jpeg",".png"]:
        continue
    image=Image.open(image_path)
    response=client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[prompt,image]
    )
    output_file=output_folder/f"{image_path.stem}.txt"
    with open(output_file,"w",encoding="utf-8") as f:
        f.write(response.text)
    print(f"Saved:{output_file.name}")

Saved:page_10_img_1.txt
Saved:page_1_img_1.txt
Saved:page_1_img_10.txt
Saved:page_1_img_11.txt
Saved:page_1_img_12.txt
Saved:page_1_img_13.txt
Saved:page_1_img_14.txt
Saved:page_1_img_2.txt
Saved:page_1_img_3.txt
Saved:page_1_img_4.txt
Saved:page_1_img_5.txt
Saved:page_1_img_6.txt
Saved:page_1_img_7.txt
Saved:page_1_img_8.txt
Saved:page_1_img_9.txt
Saved:page_2_img_1.txt
Saved:page_2_img_10.txt


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 20.395278503s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'},